# Beer Game: student personalities and extra deliberation

This is the revised **v3 Flex** experiment. It uses four fictional UB sophomore profiles, paired role assignments, known stochastic customer demand, and fixed-high GPT-6 Luna. Short submits an initial choice; adequate can revise once. Reports go to `reports/` and exact raw evidence goes to `data/`.

Start with the controls below. Opening this notebook and running review cells makes **no paid calls**. The original campaign cap is $10 including its previous $5.19360675. This is a feasibility pilot, not a powered human study. Read [the protocol](../docs/ECONOMY_PROTOCOL_20261011.md) and [the literature review](../reports/revised_design_20261011/REPORT.md).

**Cost amendment, 11 October 2026:** This notebook now uses fixed-high GPT-6 Luna Flex (v3), with a **$2.20** run cap. Together with the **$0.7198179** Mini calibration already spent, this attempt stays below **$2.92**. The original $10 campaign guard also remains. Mini calibration failed its held-out revision-completion check; it cannot be reused for Luna. [Protocol](../docs/ECONOMY_PROTOCOL_20261011.md) | [Mini calibration report](../reports/anytime_api_206b831bd7eab63e/REPORT.md). All run flags default off.

## Completed pilot and live progress

The 11 October Luna pilot is complete: **2/2 chains, 160 model orders, zero fallbacks**, passed calibration and game token separation. Estimated Luna cost: **$0.06791342**. [Readable findings](../reports/economy_api_a80275c58f2c07e0/FINDINGS.md) | [Full Word report](../reports/economy_api_a80275c58f2c07e0/Economy_Study_Report.docx) | [Workbook](../reports/economy_api_a80275c58f2c07e0/DETAILS.xlsx).

**To monitor a background run:** set `WATCH_RUNNING=True`, keep all `RUN_...` flags false, and run the cells. The watcher shows week, active requests, completed chains and charged/reserved dollars without buying calls. For this completed pilot it will show complete. When starting new games in this notebook, the game cell shows progress automatically.

**To reuse the passed calibration for new games:** set `MODE='api'`, `RUN_CALIBRATION=False`, `RUN_GAMES=True`, `WATCH_RUNNING=False`, choose a new `REPLICATE_ID`, and set `CALIBRATION_RUN = ROOT/'data/runs/economy_api_a80275c58f2c07e0'` in Controls. Choose `PAIRS` and `WEEKS` before running; keep the campaign/dollar guards. Multiples of four balance profile-role assignments. One pilot pair is descriptive, and eight pairs is only the reporting floor. The default seed repeats indexed demand paths across run IDs; new IDs buy new model draws. Enter the key only in the masked prompt.


In [1]:
from pathlib import Path
import sys, json, os, getpass, asyncio
from dataclasses import replace
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src/beergame_study').exists():
    raise RuntimeError('Open this notebook from the beergame project.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from beergame_study.economy_protocol import EconomyConfig
from beergame_study.shared_budget import run_economy_study as run_study
from beergame_study.economy_report import analyze, build_report, planned_pairs
from beergame_study.student_profiles import profiles, assignment
from beergame_study.notebook_progress import NotebookProgress
from beergame_study.operational_io import install_windows_file_retry
install_windows_file_retry()
print('Project:', ROOT)

Project: d:\beergame


## Controls

Set a run flag to `True` only for the stage you intend to execute. Keep `MODE='offline'` for free software checks. Set `MODE='api'` explicitly for paid calibration/games. Calibration is 16 scenarios (32 calls at most), not full games. A failed live screen prevents purchasing games.

The 24-week setting is a prespecified inexpensive feasibility game: four practice weeks, 20 model weeks, then 12 automatic scoring weeks. Use 40 or more for a separately planned larger study. Choose the horizon and independent pair count before outcomes. Increase pairs in multiples of four to balance personality assignments across roles. Do not treat four roles or weekly calls as independent students.

In [2]:
MODE = 'offline'
RUN_OFFLINE_CHECK = False
RUN_CALIBRATION = False
RUN_GAMES = False
WATCH_RUNNING = True
WEEKS = 24
PAIRS = 1
RUN_CAP_USD = 2.20
CAMPAIGN_ID = '20261011'  # Existing authorization includes all earlier spending.
REPLICATE_ID = 'economy-pilot-01'  # New independent games require a new ID.
CALIBRATION_RUN = None  # Optional compatible completed data/runs/... directory.
REPORT_RUN = None       # Optional saved run for analysis; no key needed.
config = EconomyConfig(mode=MODE, weeks=WEEKS, groups_per_condition=PAIRS,
    budget_usd=RUN_CAP_USD, campaign_id=CAMPAIGN_ID, replicate_id=REPLICATE_ID)
config.validate()
print('Model:', config.model, '| Effort: high | Temperature: omitted, held constant')
print('Short: initial <= 16384 tokens; adequate: same initial + revision <= 8192')
print('Actual combined tokens are checked; these are ceilings, not guaranteed usage.')
display(pd.DataFrame(profiles().values()))
display(pd.DataFrame([dict(pair=b, **dict(zip(('retailer','wholesaler','distributor','factory'), assignment(config.seed,b)))) for b in range(PAIRS)]))
print('Flex only; calibration cap $0.18; full run cap $2.20; existing Mini calibration $0.7198179.')
print('The four-state quick check precedes full held-out calibration; failed gates buy no games.')

Model: gpt-6-luna | Effort: high | Temperature: omitted, held constant
Short: initial <= 16384 tokens; adequate: same initial + revision <= 8192
Actual combined tokens are checked; these are ceilings, not guaranteed usage.


,id,label,status,background,description
0,methodical_calm,"Methodical, emotionally steady",Fictional experimental profile; not measured f...,Second-year undergraduate business student at ...,"In ordinary coursework you are organized, depe..."
1,methodical_concerned,"Methodical, uncertainty-sensitive",Fictional experimental profile; not measured f...,Second-year undergraduate business student at ...,"In ordinary coursework you are organized, depe..."
2,flexible_calm,"Flexible, emotionally steady",Fictional experimental profile; not measured f...,Second-year undergraduate business student at ...,"In ordinary coursework you are flexible, infor..."
3,flexible_concerned,"Flexible, uncertainty-sensitive",Fictional experimental profile; not measured f...,Second-year undergraduate business student at ...,"In ordinary coursework you are flexible, infor..."


,pair,retailer,wholesaler,distributor,factory
0,0,flexible_concerned,methodical_concerned,flexible_calm,methodical_calm


Flex only; calibration cap $0.18; full run cap $2.20; existing Mini calibration $0.7198179.
The four-state quick check precedes full held-out calibration; failed gates buy no games.


In [3]:
async def new_client():
    if MODE != 'api':
        return None
    from openai import AsyncOpenAI
    key = os.environ.get('OPENAI_API_KEY') or getpass.getpass('OpenAI API key (hidden; memory only): ')
    if not key or not key.strip():
        raise RuntimeError('No key entered; no paid calls made.')
    client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=config.timeout_s)
    key = None
    return client

if RUN_OFFLINE_CHECK:
    offline_config = replace(config, mode='offline', replicate_id=REPLICATE_ID+'-software')
    OFFLINE_RUN = await run_study(offline_config, root=ROOT, on_progress=NotebookProgress())
    print('SYNTHETIC software report:', ROOT/'reports'/OFFLINE_RUN.name/'REPORT.md')
else:
    print('Offline campaign not started. No calls made.')

Offline campaign not started. No calls made.


## Calibration

This measures initial-choice completion, completed revisions, and actual total computation on early/middle/late histories and all four profiles. It never selects settings using order sizes or hypothesis direction. The progress bar below displays stage, active requests and actual charges/reservations.

In [ ]:
if RUN_CALIBRATION:
    client = await new_client()
    try:
        CALIBRATION_RUN = await run_study(config, client, root=ROOT, calibration_only=True,
                                         on_progress=NotebookProgress())
    finally:
        if client is not None:
            await client.close()
    display(Markdown((ROOT/'reports'/CALIBRATION_RUN.name/'REPORT.md').read_text(encoding='utf-8')))
else:
    print('Calibration not started. Set RUN_CALIBRATION=True to run the selected MODE.')

## Paired games

A compatible saved calibration is reused without new calibration purchases. The exact full initial prompt is shared only within a paired week/role when both histories match. Once histories differ, each condition buys its own initial choice. No reuse across independent pairs. A forecast and per-request dollar guard limit spending; budget stops still produce partial reports.

In [ ]:
if RUN_GAMES:
    client = await new_client()
    try:
        GAME_RUN = await run_study(config, client, root=ROOT,
            calibration_run=CALIBRATION_RUN, on_progress=NotebookProgress())
        REPORT_RUN = GAME_RUN
    finally:
        if client is not None:
            await client.close()
    print('Report folder:', ROOT/'reports'/GAME_RUN.name)
else:
    print('Games not started. Set RUN_GAMES=True to run the selected MODE.')

## Watch a run started from PowerShell

Use this instead of the run cells when a background process is already active. It only reads progress. Interrupt the cell to stop watching; this does not cancel the experiment.

In [ ]:
if WATCH_RUNNING:
    render = NotebookProgress()
    while True:
        pointer = ROOT/'reports/ACTIVE_ECONOMY_RUN.json'
        if not pointer.exists():
            print('No revised run is active yet.')
            break
        active = json.loads(pointer.read_text(encoding='utf-8'))
        status = json.loads(Path(active['progress']).read_text(encoding='utf-8'))
        render(status)
        if status['phase'] in ('complete', 'stopped'):
            REPORT_RUN = Path(active['run'])
            break
        await asyncio.sleep(2)
else:
    print('Progress watcher disabled.')

## Review results and hypothesis checks (no API calls)

This displays the primary bullwhip result, cost result and matched-state supply-line sensitivity. The original structural beta fits remain exploratory. Unusable coefficients stay flagged. One pair cannot support an inferential conclusion. Synthetic results are explicitly labeled.

In [ ]:
if REPORT_RUN is None:
    pointer = ROOT/'reports/LATEST_ECONOMY_REPORT.json'
    if pointer.exists():
        REPORT_RUN = Path(json.loads(pointer.read_text(encoding='utf-8'))['run'])
if REPORT_RUN is not None and (Path(REPORT_RUN)/'progress.json').exists():
    status = json.loads((Path(REPORT_RUN)/'progress.json').read_text(encoding='utf-8'))
    if status['phase'] not in ('complete', 'stopped'):
        print('Run is active; use the watcher before exporting its report.')
    else:
        report_folder = build_report(REPORT_RUN, root=ROOT)
        result = analyze(REPORT_RUN)
        print('Report:', report_folder/'Economy_Study_Report.docx')
        print('Workbook:', report_folder/'DETAILS.xlsx')
        display(result['hypotheses'])
        display(result['groups'])
        display(result['roles'])
        if (report_folder/'orders.png').exists():
            display(Image(filename=str(report_folder/'orders.png')))
else:
    print('No revised run exists yet. The literature/design review is under reports/revised_design_20261011/.')

## Plan replication and costs

The following is an **illustration under statistical assumptions**, not a promise of significance or a recommended effect size. The effect is measured in standard deviations of independent paired-chain differences. Meaningful effects and variance estimates must justify the final sample plan. Eight pairs is only the software inference floor. A new campaign cap requires your explicit authorization.

In [ ]:
display(pd.DataFrame([planned_pairs(d) for d in (0.5, 0.8, 1.0)]))
print('Cost controls: within-pair initial sharing, prompt-prefix caching, exact resume, saved calibration reuse, free analysis.')
print('Batch API can be cheaper but is not implemented in this interactive runner; sequential games can be slow in batch.')